In [ ]:
#Step 1:h5 file creation for each timestep
import h5py
import numpy as np
import os

def create_h5_for_each_timestep(original_file_path, output_folder):
    # Open the original HDF5 file to read the data
    with h5py.File(original_file_path, 'r') as hdf:
        u = hdf['u']  # Shape 
        v = hdf['v']  # Shape 
        w = hdf['w']  # Shape 
        u_max = hdf['u_max']  # Shape (101,)
        v_max = hdf['v_max']  # Shape (101,)
        w_max = hdf['w_max']  # Shape (101,)
        mask = hdf['mask'][...]  # Copy mask as is
        dx = hdf['dx'][...]  # Copy dx as is

        # Ensure the output folder exists
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        # Loop over each timestep (change according to your timesteps)
        for i in range(99):
            # Create a new HDF5 file for each timestep
            output_file_path = os.path.join(output_folder, f'timestep_{i:05d}.h5')
            with h5py.File(output_file_path, 'w') as hdf_out:
                # Store the velocity arrays for the current timestep
                hdf_out.create_dataset('u', data=u[i][np.newaxis, ...])  # Shape (1, 50, 36, 57)
                hdf_out.create_dataset('v', data=v[i][np.newaxis, ...])  # Shape (1, 50, 36, 57)
                hdf_out.create_dataset('w', data=w[i][np.newaxis, ...])  # Shape (1, 50, 36, 57)

                # Store the max value arrays for the current timestep
                hdf_out.create_dataset('u_max', data=u_max[i][np.newaxis])  # Shape (1,)
                hdf_out.create_dataset('v_max', data=v_max[i][np.newaxis])  # Shape (1,)
                hdf_out.create_dataset('w_max', data=w_max[i][np.newaxis])  # Shape (1,)

                # Copy the mask and dx components as is
                hdf_out.create_dataset('mask', data=mask)
                hdf_out.create_dataset('dx', data=dx)

            print(f"Created file for timestep {i:05d}: {output_file_path}")

# File paths
original_file_path = r"Your Path\velocity_data_simv_v3.h5"
output_folder = r"Your Path\Step_2_Downsampling\Support_files"

# Create HDF5 files for each timestep
create_h5_for_each_timestep(original_file_path, output_folder)

In [ ]:
#Step 2: Downsampling Application in Each H5 Timestep File
import h5py
import numpy as np
import os
import random
import fft_downsampling as fft
import scipy.ndimage as ndimage
from h5functions import save_to_h5

def process_downsampling_for_timestep(input_filepath, output_filepath, downsample, mag_values, crop_ratio):
    # Load the mask once
    with h5py.File(input_filepath, mode='r') as hf:
        mask = np.asarray(hf['mask'][0])
        data_count = len(hf.get("u"))

    is_mask_saved = False  # Track whether the mask is saved to avoid redundant writes

    for idx in range(data_count):
        targetSNRdb = 15.5 #Replace with your SNRdb
        print("Processing data row", idx, "target SNR", targetSNRdb, "db")

        # Create the magnitude based on the possible values (augmentation)
        mag_multiplier = mag_values[idx % len(mag_values)]
        mag_image = mask * mag_multiplier

        # Load the velocity U V W from H5
        with h5py.File(input_filepath, mode='r') as hf:
            hr_u = np.asarray(hf['u'][idx])
            hr_v = np.asarray(hf['v'][idx])
            hr_w = np.asarray(hf['w'][idx])

        # Set VENC values to a specific value of your data
        venc_u, venc_v, venc_w = 1.8, 1.8, 1.8
        print(f"Using fixed VENC: {venc_u} m/s for all components")

        # Perform the downsampling
        lr_u, mag_u = fft.downsample_phase_img(hr_u, mag_image, venc_u, crop_ratio, targetSNRdb)
        lr_v, mag_v = fft.downsample_phase_img(hr_v, mag_image, venc_v, crop_ratio, targetSNRdb)
        lr_w, mag_w = fft.downsample_phase_img(hr_w, mag_image, venc_w, crop_ratio, targetSNRdb)

        # Save the downsampled images
        save_to_h5(output_filepath, "u", lr_u)
        save_to_h5(output_filepath, "v", lr_v)
        save_to_h5(output_filepath, "w", lr_w)

        save_to_h5(output_filepath, "mag_u", mag_u)
        save_to_h5(output_filepath, "mag_v", mag_v)
        save_to_h5(output_filepath, "mag_w", mag_w)

        save_to_h5(output_filepath, "venc_u", venc_u)
        save_to_h5(output_filepath, "venc_v", venc_v)
        save_to_h5(output_filepath, "venc_w", venc_w)
        save_to_h5(output_filepath, "SNRdb", targetSNRdb)

        # Only save the mask once
        if not is_mask_saved:
            new_mask = ndimage.zoom(mask, crop_ratio, order=1)
            print("Saving downsampled mask...")
            save_to_h5(output_filepath, "mask", new_mask)
            is_mask_saved = True

def main():
    # Configurations
    base_path = r"Your Path\Step_2_Downsampling"
    input_folder = os.path.join(base_path, 'Support_files')
    output_folder = os.path.join(base_path, 'Support_files_downsampled')

    # Ensure output directory exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Downsample configuration
    downsample = 2
    crop_ratio = 1 / downsample

    # Magnitude values
    mag_values = np.asarray([120])  # in px values [0-4095]

    # Process each HDF5 file (adjust according to timesteps)
    for timestep_idx in range(99):
        input_filepath = os.path.join(input_folder, f'timestep_{timestep_idx:05d}.h5')
        output_filepath = os.path.join(output_folder, f'downsampled_timestep_{timestep_idx:05d}.h5')

        print(f"\nProcessing file: {input_filepath}")
        process_downsampling_for_timestep(input_filepath, output_filepath, downsample, mag_values, crop_ratio)

    print("Downsampling complete for all timesteps.")

if __name__ == '__main__':
    main()

In [ ]:
#Step 3:Combinantion_of_all_downsampled_h5_files_in_one
import h5py
import numpy as np
import os

# Define paths
input_folder = r"Your Path\Step_2_Downsampling\Support_files_downsampled"
output_file = r"Your Path\Step_2_Downsampling\downsampled_simv_v3.h5"

# Number of timesteps and shape of arrays (Adjust Accordingly)
num_timesteps = 99 
array_shape = (36, 96, 96)

# Initialize arrays for the consolidated data
SNRdb_array = np.zeros((num_timesteps,))
mag_u_array = np.zeros((num_timesteps,) + array_shape)
mag_v_array = np.zeros((num_timesteps,) + array_shape)
mag_w_array = np.zeros((num_timesteps,) + array_shape)
u_array = np.zeros((num_timesteps,) + array_shape)
v_array = np.zeros((num_timesteps,) + array_shape)
w_array = np.zeros((num_timesteps,) + array_shape)
venc_u_array = np.zeros((num_timesteps,))
venc_v_array = np.zeros((num_timesteps,))
venc_w_array = np.zeros((num_timesteps,))

# Load data from each HDF5 file and fill the arrays
for idx in range(num_timesteps):
    file_path = os.path.join(input_folder, f"downsampled_timestep_{idx:05d}.h5")
    
    with h5py.File(file_path, 'r') as h5_file:
        # Read and store data from each file
        SNRdb_array[idx] = h5_file['SNRdb'][()]
        mag_u_array[idx] = h5_file['mag_u'][()]
        mag_v_array[idx] = h5_file['mag_v'][()]
        mag_w_array[idx] = h5_file['mag_w'][()]
        u_array[idx] = h5_file['u'][()]
        v_array[idx] = h5_file['v'][()]
        w_array[idx] = h5_file['w'][()]
        venc_u_array[idx] = h5_file['venc_u'][()]
        venc_v_array[idx] = h5_file['venc_v'][()]
        venc_w_array[idx] = h5_file['venc_w'][()]

# For the mask, use the data from timestep 47 (change if to something else if you like) or calculate the mean mask
mask_file_path = os.path.join(input_folder, "downsampled_timestep_00048.h5")
with h5py.File(mask_file_path, 'r') as h5_file:
    mask = h5_file['mask'][()]

# Save all the consolidated data into a new HDF5 file
with h5py.File(output_file, 'w') as output_h5:
    output_h5.create_dataset('SNRdb', data=SNRdb_array)
    output_h5.create_dataset('mag_u', data=mag_u_array)
    output_h5.create_dataset('mag_v', data=mag_v_array)
    output_h5.create_dataset('mag_w', data=mag_w_array)
    output_h5.create_dataset('u', data=u_array)
    output_h5.create_dataset('v', data=v_array)
    output_h5.create_dataset('w', data=w_array)
    output_h5.create_dataset('venc_u', data=venc_u_array)
    output_h5.create_dataset('venc_v', data=venc_v_array)
    output_h5.create_dataset('venc_w', data=venc_w_array)
    output_h5.create_dataset('mask', data=mask)

print(f"Consolidated HDF5 file saved at {output_file}")